In [10]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [11]:
train_transaction = pd.read_csv('ieee-fraud-detection/train_transaction.csv')
train_identity = pd.read_csv('ieee-fraud-detection/train_identity.csv')
test_transaction = pd.read_csv('ieee-fraud-detection/test_transaction.csv')
test_identity = pd.read_csv('ieee-fraud-detection/test_identity.csv')
train = train_transaction.merge(train_identity, on='TransactionID', how = 'left')
test = test_transaction.merge(test_identity, on='TransactionID', how = 'left')

In [15]:
### delete_unused_cols
delete_cols = ['id_29', 'V191', 'V196', 'V28', 'V241', 'V107', 'V117', 'V119', 'V120', 'V113', 'V305', 'V88', 'V89', 'V27', 'V41', 'V65', 'V325', 'V327', 'V68', 'id_16', 'id_27']
cat_fea = ['ProductCD','card1','card2','card3','card4','card5','card6','addr1','addr2','P_emaildomain','R_emaildomain','DeviceType','DeviceInfo']

for i in range(1,10):
    if 'M'+str(i) not in delete_cols:
        cat_fea.append('M'+str(i))
for i in range(12,39):
    if 'id_'+str(i) not in delete_cols:
        cat_fea.append('id_'+str(i))

num_fea = [] #46
for fea in train.columns:
    if fea not in ['TransactionID','isFraud'] and fea not in cat_fea and fea not in delete_cols:
        num_fea.append(fea)

In [16]:
test.columns  = test.columns.str.replace('-', '_')
# define features — everything except ID and label
features = num_fea + cat_fea

# time based split — 80% train, 20% validation
split = train['TransactionDT'].quantile(0.80)
tr = train[train['TransactionDT'] <= split].copy()
val = train[train['TransactionDT'] > split].copy()

process categorical feature:
1. unique value < 64  .astype('category')
2. unique_value >64 handle with other feature. like calculate frequency
3. for important categorical, analysis the distribution first 

In [17]:
n = 64

for col in cat_fea:
    cur_unique = tr[col].nunique()

    if cur_unique <= n:
        tr[col] = tr[col].astype('category')
        val[col] = val[col].astype('category')
        test[col] = test[col].astype('category')

    else:
        # frequency encoding — computed on tr only, mapped to val/test (no leakage)
        freq_map = tr[col].value_counts()
        tr[col + '_freq'] = tr[col].map(freq_map)
        val[col + '_freq'] = val[col].map(freq_map)
        test[col + '_freq'] = test[col].map(freq_map)

        # target encoding — fraud count per category, computed on tr only
        target_freq_map = tr[tr['isFraud'] == 1].groupby(col)['isFraud'].count()
        tr[col + '_target_freq'] = tr[col].map(target_freq_map)
        val[col + '_target_freq'] = val[col].map(target_freq_map)
        test[col + '_target_freq'] = test[col].map(target_freq_map)

        # fraud rate per category = fraud count / total count
        tr[col + '_target_ratio'] = tr[col + '_target_freq'] / tr[col + '_freq']
        val[col + '_target_ratio'] = val[col + '_target_freq'] / val[col + '_freq']
        test[col + '_target_ratio'] = test[col + '_target_freq'] / test[col + '_freq']

        # remove original column from cat_fea, add encoded cols to num_fea
        num_fea.extend([col + '_freq', col + '_target_freq', col + '_target_ratio'])

# rebuild final feature list
features = num_fea + cat_fea


### TransactionDT & TransactionAmt feature engineering

In [18]:
import warnings
warnings.filterwarnings("ignore")

for df in [tr, val, test]:
    # TransactionDT — cyclic time features
    df["hour"]          = (df["TransactionDT"] // 3600) % 24
    df["dayofweek"]     = (df["TransactionDT"] // 86400) % 7
    df["day"]           = df["TransactionDT"]  // 86400
    df["hour_sin"]      = np.sin(2 * np.pi * df["hour"]      / 24)
    df["hour_cos"]      = np.cos(2 * np.pi * df["hour"]      / 24)
    df["dayofweek_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
    df["dayofweek_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)

    # TransactionAmt — amount features
    df["log_amt"]       = np.log1p(df["TransactionAmt"])
    df["amt_decimal"]   = (df["TransactionAmt"] % 1).round(2).astype("category")
    df["amt_is_round"]  = (df["TransactionAmt"] % 1 == 0).astype(np.int8)
    df["amt_last_digit"] = pd.Series(np.where(
        df["TransactionAmt"] % 1 == 0,
        df["TransactionAmt"].astype(int) % 10,
        -1
    ), index=df.index).astype("category")

# update feature lists
new_num = ["hour", "dayofweek", "day", "hour_sin", "hour_cos",
           "dayofweek_sin", "dayofweek_cos", "log_amt", "amt_is_round"]
new_cat = ["amt_decimal", "amt_last_digit"]

num_fea.extend(new_num)
cat_fea.extend(new_cat)
print(f"Added {len(new_num)} numeric and {len(new_cat)} category features")

Added 9 numeric and 2 category features


### Log transform high-skew numeric features (|skew| > 10)

In [19]:
# compute skew on tr only
skew_df = (tr[num_fea]
           .select_dtypes(include=[np.number])
           .skew()
           .reset_index()
           .rename(columns={"index": "feature", 0: "skewness"}))

high_skew_cols = skew_df.loc[skew_df["skewness"].abs() > 10, "feature"].tolist()
print(f"Features to log transform ({len(high_skew_cols)}): {high_skew_cols}")

for df in [tr, val, test]:
    for col in high_skew_cols:
        if col in df.columns:
            df[col] = np.log1p(df[col].clip(lower=0))

print("Done.")

Features to log transform (199): ['TransactionAmt', 'C1', 'C2', 'C3', 'C4', 'C6', 'C7', 'C8', 'C10', 'C11', 'C12', 'C14', 'V1', 'V14', 'V23', 'V37', 'V38', 'V44', 'V45', 'V47', 'V55', 'V56', 'V77', 'V78', 'V86', 'V87', 'V96', 'V99', 'V100', 'V101', 'V102', 'V103', 'V104', 'V105', 'V106', 'V108', 'V109', 'V110', 'V111', 'V112', 'V114', 'V116', 'V118', 'V121', 'V122', 'V123', 'V125', 'V126', 'V127', 'V128', 'V129', 'V130', 'V131', 'V132', 'V133', 'V134', 'V135', 'V136', 'V137', 'V138', 'V142', 'V161', 'V162', 'V163', 'V166', 'V167', 'V168', 'V172', 'V176', 'V177', 'V178', 'V179', 'V180', 'V182', 'V183', 'V186', 'V187', 'V190', 'V192', 'V193', 'V198', 'V199', 'V200', 'V201', 'V202', 'V203', 'V204', 'V205', 'V206', 'V207', 'V208', 'V209', 'V210', 'V211', 'V212', 'V213', 'V214', 'V215', 'V216', 'V218', 'V219', 'V220', 'V221', 'V222', 'V223', 'V224', 'V225', 'V226', 'V227', 'V228', 'V229', 'V230', 'V231', 'V232', 'V236', 'V240', 'V242', 'V243', 'V244', 'V245', 'V246', 'V247', 'V248', 'V249',

### Bin D15

In [20]:
# bin edges from tr only — applied to val/test (no leakage)
n_bins = 10
_, bin_edges = pd.qcut(tr["D15"], q=n_bins, retbins=True, duplicates="drop")

for df in [tr, val, test]:
    df["D15_bin"] = pd.cut(df["D15"], bins=bin_edges, labels=False,
                            include_lowest=True).astype("category")

cat_fea.append("D15_bin")
num_fea.remove("D15")

# rebuild features
features = num_fea + cat_fea
print(f"D15 binned into {tr['D15_bin'].nunique()} bins")
print(f"Total features: {len(features)}")

D15 binned into 8 bins
Total features: 470
